In [1]:
import sys
sys.path.append('../../1_figure_CL_proof_of_concept/code/')
import utils_00 as gf_utils

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from itertools import combinations
import scanpy as sc
import os

large_data_dir = gf_utils.large_data_dir

# Load probe reads and pcr swap likelihoods

In [ ]:
feature_set = '../../3_figure_FFPE/code/00_probabilistic_genotype_assignment/feature_set_all.csv'

lib = 'GBM'
BC = 'BC002'

sample = BC + '_' + lib
alt_gapfill_set = pd.read_csv(feature_set)
### 
alt_gapfill_set.loc[alt_gapfill_set['name'] == 'EGFR VIII','gapfill_from_transcriptome'] = 'GGC' ## temporary fix for EGFR VIII wt probe
### 
gf_dirs = {}

### first get probe_reads to use for the patient
adata_path = large_data_dir + 'GBM_' + BC + '_genotyped.h5ad' ### this is just for cell names; it would not usually use the genotyped h5ad because it wouldn't be generated yet, but cell names are the same so we use it here to not have to upload a redundant h5ad just for this step
gf_dir = large_data_dir + 'gf_decrosslink_4plex/BC' + str(int(BC.replace('BC',''))) + '_giftwrap/'

min_percent_supporting = 0.9
collapse_across_probes = True
probe_reads = gf_utils.get_input_probe_reads(gf_dir, read_threshold = 0, adata_path = adata_path, min_percent_supporting=min_percent_supporting, collapse_across_probes=collapse_across_probes)

###
probe_reads.loc[(probe_reads['probe_idx'] == 21) & (probe_reads['gapfill'] == 'GGC'),'probe_idx'] = 383  ### temporary fix for EGFR VIII control probe idx
###

gapfill_likelihoods = pd.read_csv('../../3_figure_FFPE/code/00_probabilistic_genotype_assignment/output/likelihood_tables_with_null_GBM/' + BC + '_gapfill_likelihoods.csv')
columns_to_rename = gapfill_likelihoods.columns.difference(['probe_idx', 'gapfill'])
gapfill_likelihoods.rename(columns={col: 'p_gapfill_given_' + col for col in columns_to_rename}, inplace=True)
###
gapfill_likelihoods = gapfill_likelihoods.loc[:,gapfill_likelihoods.columns.str.len() < 100] ### temporary fix for EGFR VIII control probe idx
###

pcr_swap_likelihoods = pd.read_csv('../../3_figure_FFPE/code/00_probabilistic_genotype_assignment/output/likelihood_tables_GBM/patient_' + BC + '_swap_probabilities.csv')
pcr_swap_likelihoods.rename(columns={'likelihood':'no_pcr_swap_likelihood'}, inplace=True)
pcr_swap_likelihoods['pcr_swap_likelihood'] = 1 - pcr_swap_likelihoods['no_pcr_swap_likelihood']

if 'gapfill' not in gapfill_likelihoods.columns:
    gapfill_likelihoods['gapfill'] = np.nan

variant_probes = gapfill_likelihoods.loc[gapfill_likelihoods['gapfill'].notna(),'probe_idx'].unique()

probe_reads = probe_reads.loc[probe_reads['probe_idx'].isin(variant_probes)]
probe_reads = probe_reads.merge(gapfill_likelihoods.loc[gapfill_likelihoods['gapfill'].notna()], how='left', on = ['probe_idx','gapfill'])

probe_reads = probe_reads.merge(pcr_swap_likelihoods[['pcr_duplicate_count','pcr_swap_likelihood']], how='left', on = 'pcr_duplicate_count')
probe_reads.drop(['umi','percent_supporting','probe_barcode'],axis=1, inplace=True)


819118 UMIs found
Collapsing UMIs across probes, 819118 UMIs remaining (100.00%)
Filtering probe reads based on read threshold (0) and min percent supporting (0.9), 792315 UMIs remaining (96.73%)
Filtering cells based on min counts (0) and genes (0) in WTA
Filtering probe reads based on cell barcodes in adata, 755337 UMIs remaining (92.21%)


# get allele probabilities

In [3]:
manifest = gf_utils.get_manifest(gf_dir)

## select UMIs with low pcr swap likelihood and not probe 383 (EGFR VIII) because wt probe is different probe pair
subset_pr = probe_reads.loc[~(probe_reads['probe_idx'].isin([383])) & (probe_reads['pcr_swap_likelihood'] < 0.1)].copy()
for probe_idx in subset_pr['probe_idx'].unique():
    wt_gapfill = manifest.loc[probe_idx,'gapfill_from_transcriptome']
    mut_gapfill = manifest.loc[probe_idx,'gap_probe_sequence']
    subset_pr.loc[subset_pr['probe_idx'] == probe_idx, 'p_gapfill_given_wt'] = subset_pr.loc[subset_pr['probe_idx'] == probe_idx, 'p_gapfill_given_' + wt_gapfill]
    subset_pr.loc[subset_pr['probe_idx'] == probe_idx, 'p_gapfill_given_mut'] = subset_pr.loc[subset_pr['probe_idx'] == probe_idx, 'p_gapfill_given_' + mut_gapfill]
    subset_pr.loc[(subset_pr['probe_idx'] == probe_idx) & (subset_pr['gapfill'] == wt_gapfill), 'initial_assignment'] = 'wt'
    subset_pr.loc[(subset_pr['probe_idx'] == probe_idx) & (subset_pr['gapfill'] == mut_gapfill), 'initial_assignment'] = 'mut'

subset_pr['p_wt'] = subset_pr['p_gapfill_given_wt'] / (subset_pr['p_gapfill_given_wt'] + subset_pr['p_gapfill_given_mut'])
subset_pr['p_mut'] = subset_pr['p_gapfill_given_mut'] / (subset_pr['p_gapfill_given_wt'] + subset_pr['p_gapfill_given_mut'])


# Count assigned and unassigned; save

In [4]:
n_unassigned_pre = (subset_pr['initial_assignment'].isna().sum())
n_assigned_post = ((subset_pr['initial_assignment'].isna()) & (subset_pr[['p_wt','p_mut']].max(axis=1) >= 0.9)).sum()
n_unassigned_post = ((subset_pr['initial_assignment'].isna()) & (subset_pr[['p_wt','p_mut']].max(axis=1) < 0.9)).sum()

print(n_unassigned_pre / len(subset_pr), n_assigned_post / len(subset_pr), n_unassigned_post / len(subset_pr))

0.020714608774310268 0.018995929443690638 0.0017186793306196292


In [5]:
pd.DataFrame({'assignment_status': ['unassigned_pre', 'assigned_post', 'unassigned_post'], 'proportion': [n_unassigned_pre / len(subset_pr), n_assigned_post / len(subset_pr), n_unassigned_post / len(subset_pr)]}).to_csv('../output/GBM_gapfill_assignment_metrics.csv')